## Librairies and functions

In [ ]:
from model import Model

In [ ]:
from settings import Config
from config_species import get_settings
import matplotlib.pyplot as plt
from pathlib import Path
import pickle

In [ ]:
def load_dataset(species_folder, species, dataset_type="train", method_compression=None, parameter_compression=None) -> tuple:
        """Loads the dataset from the save path"""
        # Load the dataset
        if method_compression!=None:
            with open(Path(species_folder, dataset_type, species + "_X_" + dataset_type + "_" + method_compression + "_"+ parameter_compression + ".pkl"), "rb") as f:
                X = pickle.load(f)
        else : 
            print("load baseline")
            with open(Path(species_folder, dataset_type, species + "_X_" + dataset_type + ".pkl"), "rb") as f:
                X = pickle.load(f)


        with open(Path(species_folder, dataset_type, species + "_Y_" + dataset_type + ".pkl"), "rb") as f:
            Y = pickle.load(f)
        
        print("Dataset loaded from " + str(species_folder) + f"/{dataset_type}")
        return X, Y



## Parameters

In [ ]:
selected_species = "thyolo" #"thyolo" # gibbon / ptw
settings = get_settings(selected_species)
config = Config(settings) # Pass settings to your config object
print("Loaded settings for:", selected_species)
print(config.preprocessing.dict())
print(config.data.dict())

In [ ]:
config.data.species_folder=r"c:\Users\loren\Documents\Postdoc\Compressed_sensing\Data\Thyolo"
config.preprocessing.audio_extension=".npy"

method_compression="cs"
parameter_compression="0.2"

In [ ]:
if method_compression!=None : 
    save_path=Path(config.data.species_folder, "results", f"{config.data.positive_class}_{method_compression}_{parameter_compression}")
    save_path.mkdir(parents=True, exist_ok=True)
else : 
    save_path=Path(config.data.species_folder, "results", f"{config.data.positive_class}_baseline")
    save_path.mkdir(parents=True, exist_ok=True)
print(save_path)

#### Architecture of the model

In [ ]:
config.cnn_architecture.dict()

In [ ]:
#Thyolo : 
config.cnn_architecture.conv_layers = 2
config.cnn_architecture.fc_layers = 2
config.cnn_architecture.conv_kernel = 8
config.cnn_architecture.conv_filters = 16
config.cnn_architecture.dropout_rate = 0.3
config.cnn_architecture.fc_units = 64
config.cnn_architecture.max_pooling_size = 2


config.model.learning_rate = 0.001
config.model.num_epochs = 50
config.model.batch_size=64

In [ ]:
#PTW : 
config.cnn_architecture.conv_layers = 1
config.cnn_architecture.fc_layers = 2
config.cnn_architecture.conv_kernel = 8
config.cnn_architecture.conv_filters = 16
config.cnn_architecture.dropout_rate = 0.3
config.cnn_architecture.fc_units = 64
config.cnn_architecture.max_pooling_size = 4


config.model.learning_rate = 0.001
config.model.num_epochs = 50
config.model.batch_size=64


In [ ]:
#Gibbon : 
config.cnn_architecture.conv_layers = 1
config.cnn_architecture.fc_layers = 2
config.cnn_architecture.conv_kernel = 8
config.cnn_architecture.conv_filters = 8
config.cnn_architecture.dropout_rate = 0.5
config.cnn_architecture.fc_units = 32
config.cnn_architecture.max_pooling_size = 4


config.model.learning_rate = 0.001
config.model.num_epochs = 50
config.model.batch_size=128

In [ ]:
config.model.num_epochs=100

## Training

In [ ]:
X_train, Y_train= load_dataset(config.data.species_folder, config.data.positive_class, method_compression=method_compression, parameter_compression=parameter_compression)
X_val, Y_val=load_dataset(config.data.species_folder, config.data.positive_class, dataset_type="val",method_compression=method_compression, parameter_compression=parameter_compression)

In [ ]:
print(f"Val set size: {len(X_val)}")
print(f"Train set size: {len(X_train)}")

In [ ]:
del model

In [ ]:
model = Model(save_path,
            input_shape=(1, X_train.shape[1], X_train.shape[2]),
            architecture_args= config.cnn_architecture.dict(),
            **config.model.dict(),
        )



In [ ]:
model

In [ ]:
"""
n_neg = 7084
n_pos = 2491

class_weights = [1.0, n_neg / n_pos]  # [1.0, 2.84]
"""

if method_compression!=None:
    model_name=method_compression+"_"+parameter_compression
else : 
    model_name="baseline"
train_losses, val_losses =model.train(X_train=X_train, Y_train=Y_train, X_val=X_val, Y_val=Y_val, model_name=model_name, early_stopping=True, patience=15, min_delta=0.0005)

#### plot val and train loss

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Training Loss')
plt.plot( val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.show()

In [ ]:
baseline_metric, baseline_metric_name = model.evaluate(
            X_val=X_val, Y_val=Y_val
        )
baseline_trainable_params = model.get_number_of_parameters()
print(baseline_metric)

## Evaluation of the model

In [ ]:
from evaluation import Evaluation
from model import Model
import torch
import os

In [ ]:
selected_species = "thyolo" #"thyolo" # gibbon / ptw
settings = get_settings(selected_species)
config = Config(settings) # Pass settings to your config object
print("Loaded settings for:", selected_species)
print(config.preprocessing.dict())
print(config.data.dict())


In [ ]:
config.data.species_folder=r"c:\Users\loren\Documents\Postdoc\Compressed_sensing\Data\Thyolo"
config.preprocessing.audio_extension=".npy"

method_compression="cs"
parameter_compression="0.2"

if method_compression!=None : 
    save_path=Path(config.data.species_folder, "results", f"{config.data.positive_class}_{method_compression}_{parameter_compression}")
    model_name=method_compression+"_"+parameter_compression+"_cnn_state.pth"
else : 
    save_path=Path(config.data.species_folder, "results", f"{config.data.positive_class}_baseline")
    model_name="baseline_cnn_state.pth"
print(save_path)

In [ ]:
model_path = os.path.join(save_path, model_name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Model.load_cnn(model_path, device)

In [ ]:
overlap=0.10
nb_to_group=0
threshold=0.8
step_size=1

evaluation = Evaluation(
                species_folder=config.data.species_folder,
                settings=config,
                overlap=overlap,
                threshold=threshold,
                step_size=step_size,
                method_compression=method_compression, 
                parameter_compression=parameter_compression,
                nb_to_group=nb_to_group, 
                force_calc_amplitudes=False,
                audio_extension=config.preprocessing.audio_extension,
            )

In [ ]:
evaluation.run(model, type="val", test_type="testing_dataset")

In [ ]:
evaluation.run(model, type="val", test_type="entire_files", preprocessing_arg=True)

## Calculation of the threshold 

In [ ]:
thresholds=[0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1]
Precision=[]
Recall=[]
overlap=0.10
nb_to_group=0
for threshold in thresholds : 


    print("Threshold : ", threshold)
    evaluation = Evaluation(
                species_folder=config.data.species_folder,
                settings=config,
                overlap=overlap,
                threshold=threshold,
                method_compression=method_compression, 
                parameter_compression=parameter_compression,
                nb_to_group=nb_to_group, 
                force_calc_amplitudes=False,
                audio_extension=config.preprocessing.audio_extension,
            )
    _, matrice, _ , precision, recall= evaluation.run(model, type="val", test_type="entire_files", preprocessing_arg=True)

    Precision.append(precision)
    Recall.append(recall)


    
    

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(np.asarray(thresholds), np.asarray(Precision), color='steelblue', lw=2, label='Precision')
ax.plot(np.asarray(thresholds), np.asarray(Recall), color='coral', lw=2, label='Recall')

ax.set_xlabel('Threshold', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Precision, Recall vs Threshold', fontsize=13)
ax.set_xlim([min(thresholds), max(thresholds)])
ax.set_ylim([0, 1.05])
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('threshold_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Loop CNN training + evaluation

In [ ]:
import torch
import os
from evaluation import Evaluation
from model import Model
import shutil

In [ ]:
from settings import Config
from config_species import get_settings
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import pickle
import pandas as pd
import gc


In [ ]:
def load_dataset(species_folder, species, dataset_type="train", method_compression=None, parameter_compression=None) -> tuple:
        """Loads the dataset from the save path"""
        # Load the dataset
        if method_compression!=None:
            with open(Path(species_folder, dataset_type, species + "_X_" + dataset_type + "_" + method_compression + "_"+ parameter_compression + ".pkl"), "rb") as f:
                X = pickle.load(f)
        else : 
            with open(Path(species_folder, dataset_type, species + "_X_" + dataset_type + ".pkl"), "rb") as f:
                X = pickle.load(f)


        with open(Path(species_folder, dataset_type, species + "_Y_" + dataset_type + ".pkl"), "rb") as f:
            Y = pickle.load(f)
        
        print("Dataset loaded from " + str(species_folder) + f"/{dataset_type}/{species}_X_{dataset_type}_{method_compression}_{parameter_compression}")
        return X, Y

#### Parameters

In [ ]:
selected_species = "thyolo" #"thyolo" # gibbon / ptw
settings = get_settings(selected_species)
config = Config(settings) # Pass settings to your config object
print("Loaded settings for:", selected_species)
print(config.preprocessing.dict())
print(config.data.dict())




In [ ]:
config.data.species_folder=r"c:\Users\loren\Documents\Postdoc\Compressed_sensing\Data\Thyolo"
config.preprocessing.audio_extension=".npy"

method_compression="cs"
parameter_compression="0.2"

In [ ]:
if method_compression!=None : 
    save_path=Path(config.data.species_folder, "results", f"{config.data.positive_class}_{method_compression}_{parameter_compression}")
    save_path.mkdir(parents=True, exist_ok=True)
else : 
    save_path=Path(config.data.species_folder, "results", f"{config.data.positive_class}_baseline")
    save_path.mkdir(parents=True, exist_ok=True)
print(save_path)

In [ ]:
X_train, Y_train= load_dataset(config.data.species_folder, config.data.positive_class, method_compression=method_compression, parameter_compression=parameter_compression)
X_val, Y_val=load_dataset(config.data.species_folder, config.data.positive_class, dataset_type="val",method_compression=method_compression, parameter_compression=parameter_compression)

In [ ]:
overlap=0.10  #0.25 for ptw and gibbon 0.10 for thyolo
nb_to_group=0 #2 for ptw and gibbon 0 for thyolo
threshold=0.8

evaluation = Evaluation(
                species_folder=config.data.species_folder,
                settings=config,
                overlap=overlap,
                threshold=threshold,
                method_compression=method_compression, 
                parameter_compression=parameter_compression,
                nb_to_group=nb_to_group, 
                force_calc_amplitudes=False,
                audio_extension=config.preprocessing.audio_extension,
            )

##### architecture

In [ ]:
#Thyolo : 
config.cnn_architecture.conv_layers = 2
config.cnn_architecture.fc_layers = 2
config.cnn_architecture.conv_kernel = 8
config.cnn_architecture.conv_filters = 16
config.cnn_architecture.dropout_rate = 0.3
config.cnn_architecture.fc_units = 64
config.cnn_architecture.max_pooling_size = 2


config.model.learning_rate = 0.001
config.model.num_epochs = 100
config.model.batch_size=64

In [ ]:
#PTW : 
config.cnn_architecture.conv_layers = 1
config.cnn_architecture.fc_layers = 2
config.cnn_architecture.conv_kernel = 8
config.cnn_architecture.conv_filters = 16
config.cnn_architecture.dropout_rate = 0.3
config.cnn_architecture.fc_units = 64
config.cnn_architecture.max_pooling_size = 4


config.model.learning_rate = 0.001
config.model.num_epochs = 100
config.model.batch_size=64


##### Loop : 10 runs

In [ ]:
first_loop=0
last_loop=10

F1_score_full=[]
F1_score=[]

for i in range(first_loop, last_loop): 
    
    
    #clean gpu
    torch.cuda.empty_cache()
    gc.collect()
    
    
    print(i)
    #create folder to save model 
    #dir_out=Path(save_path, f"model_{config.data.positive_class}_{method_compression}_{parameter_compression}_{k}")
    #os.makedirs(dir_out)
    X_train, Y_train= load_dataset(config.data.species_folder, config.data.positive_class, method_compression=method_compression, parameter_compression=parameter_compression)
    X_val, Y_val=load_dataset(config.data.species_folder, config.data.positive_class, dataset_type="val",method_compression=method_compression, parameter_compression=parameter_compression)
    
    #I- Training
    #initialise the model 
    model = Model(save_path,
            input_shape=(1, X_train.shape[1], X_train.shape[2]),
            architecture_args= config.cnn_architecture.dict(),
            **config.model.dict(),
        )
    print(config.model.dict())

    

    #train
    if method_compression!= None : 
        model_name=method_compression+"_"+parameter_compression
    else : 
        model_name="baseline"
    train_losses, val_losses =model.train(X_train=X_train, Y_train=Y_train, X_val=X_val, Y_val=Y_val, model_name=model_name, early_stopping=True, patience=15, min_delta=0.0005)
    del model, model_name


    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label='Training Loss')
    plt.plot( val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    plt.savefig(Path(save_path, f"Training_and_Validation_Loss_{i}.png"))
    plt.show()
    


    # II- Evaluation
    #load the saved model
    if method_compression!= None :
        model_name=method_compression+"_"+parameter_compression+"_cnn_state.pth"
    else : 
        model_name="baseline_cnn_state.pth"
    model_path = os.path.join(save_path, model_name)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = Model.load_cnn(model_path, device)

    f1_score=evaluation.run(model, test_type="testing_dataset")[0]
    f1_score_full=evaluation.run(model, test_type="entire_files", preprocessing_arg=True)[0]

    F1_score.append(f1_score)
    F1_score_full.append(f1_score_full)
    del model, model_name, X_train, X_val, Y_train, Y_val

#save f1 score
df = pd.DataFrame({"F1_score" : np.asarray(F1_score), "F1_score_full" : np.asarray(F1_score_full)})
df.to_csv(Path(save_path, f"{config.data.positive_class}_{method_compression}_{parameter_compression}_f1score_csv.csv"), index=False)

print("F1_score mean : ", np.mean(np.asarray(F1_score)))  
print("F1_score_full  mean : ", np.mean(np.asarray(F1_score_full)))

""" 
#clean the folder with amplitudes 
if os.path.exists(evaluation.save_amplitudes_path):
    shutil.rmtree(evaluation.save_amplitudes_path)
    print(f"Folder '{evaluation.save_amplitudes_path}' and its contents deleted successfully.")
else:
    print(f"Folder '{evaluation.save_amplitudes_path}' does not exist.")
"""

#### Loop on hyperparameter + loop on 10 runs

In [ ]:
overlap=0.10
nb_to_group=0
threshold=0.8
parameters_list=["0", "6","12"]

for parameter_compression in parameters_list : 
    if method_compression!=None : 
        save_path=Path(config.data.species_folder, "results", f"{config.data.positive_class}_{method_compression}_{parameter_compression}")
        save_path.mkdir(parents=True, exist_ok=True)
    else : 
        save_path=Path(config.data.species_folder, "results", f"{config.data.positive_class}_baseline")
        save_path.mkdir(parents=True, exist_ok=True)
    print(save_path)

    
    evaluation = Evaluation(
                species_folder=config.data.species_folder,
                settings=config,
                overlap=overlap,
                threshold=threshold,
                method_compression=method_compression, 
                parameter_compression=parameter_compression,
                nb_to_group=nb_to_group, 
                force_calc_amplitudes=False,
                audio_extension=config.preprocessing.audio_extension,
            )

    first_loop=0
    last_loop=10

    F1_score_full=[]
    F1_score=[]

    for i in range(first_loop, last_loop): 
      
        #clean gpu
        torch.cuda.empty_cache()
        gc.collect()
        
        
        print(i)
        #create folder to save model 
        #dir_out=Path(save_path, f"model_{config.data.positive_class}_{method_compression}_{parameter_compression}_{k}")
        #os.makedirs(dir_out)
        X_train, Y_train= load_dataset(config.data.species_folder, config.data.positive_class, method_compression=method_compression, parameter_compression=parameter_compression)
        X_val, Y_val=load_dataset(config.data.species_folder, config.data.positive_class, dataset_type="val",method_compression=method_compression, parameter_compression=parameter_compression)
    
        
        #I- Training
        #initialise the model 
        model = Model(save_path,
                input_shape=(1, X_train.shape[1], X_train.shape[2]),
                architecture_args= config.cnn_architecture.dict(),
                **config.model.dict(),
            )

        #train
        if method_compression!= None : 
            model_name=method_compression+"_"+parameter_compression
        else : 
            model_name="baseline"
        train_losses, val_losses =model.train(X_train=X_train, Y_train=Y_train, X_val=X_val, Y_val=Y_val, model_name=model_name, early_stopping=False)
        del model, model_name

        # II- Evaluation
        #load the saved model
        if method_compression!= None :
            model_name=method_compression+"_"+parameter_compression+"_cnn_state.pth"
        else : 
            model_name="baseline_cnn_state.pth"
        model_path = os.path.join(save_path, model_name)
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = Model.load_cnn(model_path, device)

        f1_score=evaluation.run(model, test_type="testing_dataset")[0]
        f1_score_full=evaluation.run(model, test_type="entire_files", preprocessing_arg=True)[0]

        F1_score.append(f1_score)
        F1_score_full.append(f1_score_full)
        del model, model_name

    #save f1 score
    df = pd.DataFrame({"F1_score" : np.asarray(F1_score), "F1_score_full" : np.asarray(F1_score_full)})
    df.to_csv(Path(save_path, f"{config.data.positive_class}_{method_compression}_{parameter_compression}_f1score_csv.csv"), index=False)

    print("F1_score mean : ", np.mean(np.asarray(F1_score)))  
    print("F1_score_full  mean : ", np.mean(np.asarray(F1_score_full))) 
    
    
    #clean the folder with amplitudes 
    if os.path.exists(evaluation.save_amplitudes_path):
        shutil.rmtree(evaluation.save_amplitudes_path)
        print(f"Folder '{evaluation.save_amplitudes_path}' and its contents deleted successfully.")
    else:
        print(f"Folder '{evaluation.save_amplitudes_path}' does not exist.")
    
    
    del X_train, Y_train, X_val, Y_val, evaluation
    print("Next parameter")


        

c:\Users\loren\Documents\Postdoc\Compressed_sensing\Data\PTW\results\PTW_flac_0
0
Dataset loaded from c:\Users\loren\Documents\Postdoc\Compressed_sensing\Data\PTW/train/PTW_X_train_flac_0
Dataset loaded from c:\Users\loren\Documents\Postdoc\Compressed_sensing\Data\PTW/val/PTW_X_val_flac_0
CNN model state dict saved to c:\Users\loren\Documents\Postdoc\Compressed_sensing\Data\PTW\results\PTW_flac_0\flac_0_cnn_state.pth!
CNN model state dict saved to c:\Users\loren\Documents\Postdoc\Compressed_sensing\Data\PTW\results\PTW_flac_0\flac_0_cnn_state.pth!
CNN model state dict saved to c:\Users\loren\Documents\Postdoc\Compressed_sensing\Data\PTW\results\PTW_flac_0\flac_0_cnn_state.pth!
CNN model state dict saved to c:\Users\loren\Documents\Postdoc\Compressed_sensing\Data\PTW\results\PTW_flac_0\flac_0_cnn_state.pth!
CNN model state dict saved to c:\Users\loren\Documents\Postdoc\Compressed_sensing\Data\PTW\results\PTW_flac_0\flac_0_cnn_state.pth!
CNN model state dict saved to c:\Users\loren\Docum

KeyboardInterrupt: 